## Week 5: Transformers and Protein Language Models

> You shall know a word by the company it keeps.

**John Rupert Firth** a British linguist in *A Synopsis of Linguistic Theory* (1957)

The quote from John Firth above points to a unique aspect of human language. The meaning of a word can be inferred by the context in which it is commonly used. In any large corpus of text, words do not appear in any arbitrary pattern. They usually are found alongside other words. Being able to statistically model this context is a significant reason for the rapid development of Large Language Models in recent years. 

Proteins sequences, it turns out, also share broad similarities with language. Thus, many of the tools developed to model language have also been applied for protein sequences. In this week, we will look at language models as it applies to both text and proteins sequences. First, what is a protein sequence? 



### 1 : Protein sequences
- What is a protein sequence? 
Proteins are long-chain amino acids. The sequence of amino acids which make a protein is its primary structure. We use this sequence information for analysing its structur and function. 

Protein sequences are publicly accessible through the UniProtKB. In the next code cell, let us familiarise ourselves with this database and how to navigate it. Head over to [UniProt](https://www.uniprot.org/) and answer the following questions. 

In [2]:
# Some fun quiz about UniProt Database
# 1. Which of the following English Language words are present in the sequences of the UniProt Database?
# a) CRICKET
# b) PYTHON
# c) MACHINELEARNING
# d) MITOCHONDRIA
# e) KINASES
# f) AMYLOID

# SOME MORE FUN STUFF!

### 2 : Tokenisation: 
A token is the basic unit of a language model. It is a numerical quantity that represents parts of a language. It could be a character, a word, parts of a word, or phrases. What makes a good token is that it captures rich contextual information but is also frequent enough to be useful to build a vocabulary. Let us explore some options to tokenise written texts. 

- **Character-level** — each character (including spaces) is its own token

  `[E][V][E][R][Y][·][C][H][A][R][A][C][T][E][R][·][I][S][·][A][·][T][O][K][E][N]`

- **Word-level** — each word is one token

  `[EVERY][·][WORD][·][IS][·][A][·][TOKEN]`

- **Sentence-level** — the whole sentence is one token

  `[EVERY SENTENCE IS A TOKEN]`

The balance between expressivity and representation forces us to choose between tokenisation schemes. Let us use the tiny-shakespeare dataset to make some observations. 

In [3]:
with open("tinyshakespeare.txt", "r") as f:
    text = f.read()

In [8]:
# Character-level analysis
num_chars_total = len(text)
print(f"Total number of characters in the text: {num_chars_total}")

unique_chars = list(set(text))
num_unique_chars = len(unique_chars)
representation_ratio = num_chars_total / num_unique_chars

print(f"Number of unique characters in the text: {num_unique_chars}")
print(f"Representation ratio (total chars / unique chars): {representation_ratio:.2f}")
print(f"Unique characters in the text:")
print("-" * 40)
print(unique_chars)

Total number of characters in the text: 1115393
Number of unique characters in the text: 65
Representation ratio (total chars / unique chars): 17159.89
Unique characters in the text:
----------------------------------------
pcIM&Bfxj. qFXlZn:SVmARg3h-WGUeENt!TC,wur;H?YDd$OsozKQ
vbyJik'aLP


In [7]:
encode_tokens = {char: idx for idx, char in enumerate(unique_chars)}
decode_tokens = {idx: char for char, idx in encode_tokens.items()}

sample_text = "to be, or"
encoded_sample = [encode_tokens[char] for char in sample_text]
print(f"Encoded sample text: {encoded_sample}")
decoded_sample = "".join([decode_tokens[idx] for idx in encoded_sample])
print(f"Decoded sample text: {decoded_sample}")


Encoded sample text: [33, 50, 10, 56, 30, 37, 10, 50, 40]
Decoded sample text: to be, or


Note that spaces, special characters and newline characters also count for tokenisation. That is how a language model can build sensible structure into text. 

In [9]:
# Word-level analysis
import random
words = text.split()
num_words_total = len(words)
print(f"Total number of words in the text: {num_words_total}")
unique_words = list(set(words))
num_unique_words = len(unique_words)
representation_ratio_words = num_words_total / num_unique_words

print(f"Number of unique words in the text: {num_unique_words}")
print(f"Representation ratio (total words / unique words): {representation_ratio_words:.2f}")
print(f"Some unique words in the text:")
print("-" * 40)
sample_words = random.sample(unique_words, 5)
print(sample_words)

Total number of words in the text: 202651
Number of unique words in the text: 25670
Representation ratio (total words / unique words): 7.89
Some unique words in the text:
----------------------------------------
['parts.', 'slack:', "grandsire's", 'crammed', 'scarf,']


In [10]:
encode_word_tokens = {word: idx for idx, word in enumerate(unique_words)}
decode_word_tokens = {idx: word for word, idx in encode_word_tokens.items()}

sample_word_text = "to be, or"
encoded_sample_words = [encode_word_tokens[word] for word in sample_word_text.split()]
print(f"Encoded sample word text: {encoded_sample_words}")
decoded_sample_words = " ".join([decode_word_tokens[idx] for idx in encoded_sample_words])
print(f"Decoded sample word text: {decoded_sample_words}")

Encoded sample word text: [3332, 21333, 21946]
Decoded sample word text: to be, or


In [24]:
# Sentence-level analysis, assuming sentences are separated by newline.
sentences = text.split("\n")
num_sentences_total = len(sentences)
print(f"Total number of sentences in the text: {num_sentences_total}")
unique_sentences = list(set(sentences))
num_unique_sentences = len(unique_sentences)
representation_ratio_sentences = num_sentences_total / num_unique_sentences
print(f"Representation ratio (total sentences / unique sentences): {representation_ratio_sentences:.2f}")
print(f"Number of unique sentences in the text: {num_unique_sentences}")
print(f"Some unique sentences in the text:")
print("-" * 40)
sample_sentences = random.sample(unique_sentences, 3)
for sentence in sample_sentences:
    print(sentence.strip())
    print("-" * 40)


Total number of sentences in the text: 40000
Representation ratio (total sentences / unique sentences): 1.56
Number of unique sentences in the text: 25722
Some unique sentences in the text:
----------------------------------------
Was it not so?
----------------------------------------
Blue, my lord.
----------------------------------------
Is empty on the back of Montague,--
----------------------------------------


In [25]:
encode_sentence_tokens = {sentence: idx for idx, sentence in enumerate(unique_sentences)}
decode_sentence_tokens = {idx: sentence for sentence, idx in encode_sentence_tokens.items()}

sample_sentence_text = "Why, Romeo, art thou mad?"
encoded_sample_sentence = encode_sentence_tokens[sample_sentence_text]
print(f"Encoded sample sentence text: {encoded_sample_sentence}")
decoded_sample_sentence = decode_sentence_tokens[encoded_sample_sentence]
print(f"Decoded sample sentence text: {decoded_sample_sentence.strip()}")


Encoded sample sentence text: 15714
Decoded sample sentence text: Why, Romeo, art thou mad?


Clearly, the characters are the most representative. Just 65 unique characters can be represents over a million units of information in the corpus. But each character by itself does not say much. It has to be grouped into words to form some meaning. We see therefore when we group the corpus by words the total vocabulary count increased to over 25,000. Clearly less representative than characters but they gain in expressivity. Sentences are the least representative data structure. Almost all the sentences in the dataset are unique, therefore each sentence is hardly useful to learn about others. 

While looking at above numbers, it might seem that using words is a better option there is a better way to tokenise language. Many words share the same root, and therefore can be split into different parts. For instance: 

`["small", "smaller", "smallest", "big", "bigger", "biggest"]` can be better represented as: 

`["small", "big", "er", "est"] `

We limit the size of the vocabulary to increase representativeness while retaining expressivity. The common algorithm that is applied is Byte-Pair Encoding 


In [30]:
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o")   # BPE used for GPT models
tokens_bpe = enc.encode(text)
unique_bpe_tokens = list(set(tokens_bpe))
num_bpe_tokens = len(tokens_bpe)
num_bpe_tokens_unique = len(unique_bpe_tokens)
representation_ratio_bpe = num_bpe_tokens / num_bpe_tokens_unique
print(f"Total number of BPE tokens in the text: {num_bpe_tokens}")
print(f"Number of unique BPE tokens in the text: {num_bpe_tokens_unique}")
print(f"Representation ratio (total BPE tokens / unique BPE tokens): {representation_ratio_bpe:.2f}")
print(f"Some unique BPE tokens in the text:")
print("-" * 40)
sample_bpe_tokens = random.sample(unique_bpe_tokens, 5)
decoded_bpe_tokens = [enc.decode([token]) for token in sample_bpe_tokens]
print(decoded_bpe_tokens)

Total number of BPE tokens in the text: 297606
Number of unique BPE tokens in the text: 12733
Representation ratio (total BPE tokens / unique BPE tokens): 23.37
Some unique BPE tokens in the text:
----------------------------------------
[' begin', 'brief', ' lance', ' specially', ' Philip']


In [31]:
sample_text_bpe = "to be, or not to be"
encoded_sample_bpe = enc.encode(sample_text_bpe)
print(f"Encoded sample text (BPE tokens): {encoded_sample_bpe}")
decoded_sample_bpe = enc.decode(encoded_sample_bpe)
print(f"Decoded sample text from BPE tokens: {decoded_sample_bpe}")

Encoded sample text (BPE tokens): [935, 413, 11, 503, 625, 316, 413]
Decoded sample text from BPE tokens: to be, or not to be


Tokenising protein sequences follow the same ideas as that of language. For protein sequences, common method is to tokenise each amino acid. There are 22 amino acids which are used in protein sequences. Most encoding schemes use residue-level encoding for protein sequences. 

Luckily, amino acids combine the representation of characters and the expressivity of words. Each amino acid by itself contains enough information to form meaningful features that can be used for learning. 

But what exactly do we mean by *features* exactly? How can we learn this? In the next section, we learn about embeddings where meaning is encoded as vectors. Vectorising tokens allows us to mathematically explore the famous quote by John Frith shown at the beginning of this module. 

### 3 Embeddings

Embeddings are vectors which represent some meaningful information about a token. Here, *meaningful* is different from *interpretable*. A meaningful embedding only helps the network perform better on the task that it is being trained on, and not necessarily one that helps us interpret what the meaning of an embedding really is. 

In Week 3, we have already encountered embeddings. These are similar to the *node-features* that were used to model a methanol molecule. In the message passing activity, each of you had a feature vector where each dimension probed the likelihood of doing a certain action. Embeddings are conceptually similar to node-features. 

In Language models, every token can be considered a node. A core-component of a language model is the attention mechanism. In fact, as we will see in the next module the attention mechanism of language models can be considered as a special form of Graph Neural Network where each node (or a token) is connected to every other node. 


